In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF version: 2.19.0
GPU: []


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CropGuardAI', exist_ok=True)
print("Drive mounted and folder created.")

Mounted at /content/drive
Drive mounted and folder created.


In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Downloading PlantVillage dataset (~2.7 GB)...")
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset --path /content/
print("Extracting...")
!unzip -q /content/new-plant-diseases-dataset.zip -d /content/
print("Done!")


cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [00:33<00:00, 85.3MB/s]

Extracting...
Done!


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH    = 64
NUM_CLASSES = 38

train_gen = ImageDataGenerator(
    rescale=1/255., rotation_range=35,
    zoom_range=0.2, horizontal_flip=True,
    brightness_range=[0.75, 1.25], fill_mode="nearest"
).flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=True, seed=42
)

val_gen = ImageDataGenerator(rescale=1/255.).flow_from_directory(
    valid_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=False
)

print(f"Train: {train_gen.samples} images | Val: {val_gen.samples} images")

NameError: name 'train_dir' is not defined

In [ ]:
import os

# Find what actually extracted
for item in os.listdir('/content'):
    print(item)

.config
New Plant Diseases Dataset(Augmented)
test
drive
new-plant-diseases-dataset.zip
new plant diseases dataset(augmented)
sample_data


In [ ]:
import os

# Find the train folder automatically — no matter what it's named
train_dir = None
valid_dir = None

for root, dirs, files in os.walk('/content'):
    for d in dirs:
        if d.lower() == 'train' and train_dir is None:
            train_dir = os.path.join(root, d)
        if d.lower() in ('valid', 'val', 'validation') and valid_dir is None:
            valid_dir = os.path.join(root, d)

print(f"Train dir: {train_dir}")
print(f"Valid dir: {valid_dir}")
print(f"Classes found: {len(os.listdir(train_dir))}")

Train dir: /content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Valid dir: /content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
Classes found: 38


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH    = 64
NUM_CLASSES = 38

train_gen = ImageDataGenerator(
    rescale=1/255., rotation_range=35,
    zoom_range=0.2, horizontal_flip=True,
    brightness_range=[0.75, 1.25], fill_mode="nearest"
).flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=True, seed=42
)

val_gen = ImageDataGenerator(rescale=1/255.).flow_from_directory(
    valid_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=False
)

print(f"Train: {train_gen.samples} images | Val: {val_gen.samples} images")

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
Train: 70295 images | Val: 17572 images


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks

base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")

for layer in base.layers[:-50]:
    layer.trainable = False
for layer in base.layers[-50:]:
    layer.trainable = True

inp = tf.keras.Input(shape=(224,224,3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dense(512, activation="relu")(x)
x   = layers.Dropout(0.4)(x)
x   = layers.Dense(256, activation="relu")(x)
x   = layers.Dropout(0.25)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model ready.")
print(f"Trainable params: {sum(p.numpy().size for p in model.trainable_variables):,}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model ready.
Trainable params: 2,654,630


In [ ]:
import time

SAVE_PATH = "/content/drive/MyDrive/CropGuardAI/cropguard_best.keras"

cbs = [
    callbacks.ModelCheckpoint(
        SAVE_PATH, monitor="val_accuracy",
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.4,
        patience=3, min_lr=1e-7, verbose=1
    ),
]

print("Training started. Model saves to Google Drive after each improvement.\n")
start = time.time()

history = model.fit(
    train_gen, validation_data=val_gen,
    epochs=20, callbacks=cbs, verbose=1
)

elapsed = (time.time() - start) / 60
best    = max(history.history["val_accuracy"])
print(f"\nDone in {elapsed:.1f} min — Best accuracy: {best*100:.2f}%")

Training started. Model saves to Google Drive after each improvement.

Epoch 1/20
1099/1099 ━━━━━━━━━━━━━━━━━━━━ 0s 989ms/step - accuracy: 0.7629 - loss: 0.8790
Epoch 1: val_accuracy improved from None to 0.73219, saving model to /content/drive/MyDrive/CropGuardAI/cropguard_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CropGuardAI/cropguard_best.keras
1099/1099 ━━━━━━━━━━━━━━━━━━━━ 1186s 1s/step - accuracy: 0.8858 - loss: 0.3984 - val_accuracy: 0.7322 - val_loss: 1.3815 - learning_rate: 3.0000e-04
Epoch 2/20
1099/1099 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9586 - loss: 0.1334
Epoch 2: val_accuracy improved from 0.73219 to 0.80685, saving model to /content/drive/MyDrive/CropGuardAI/cropguard_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/CropGuardAI/cropguard_best.keras
1099/1099 ━━━━━━━━━━━━━━━━━━━━ 1163s 1s/step - accuracy: 0.9613 - loss: 0.1263 - val_accuracy: 0.8069 - val_loss: 1.0361 - learning_rate: 3.0000e-04
Epoch 3/20
1099/109

In [ ]:
import json

EXPORT_DIR = "/content/drive/MyDrive/CropGuardAI/plant_disease_model"
INDEX_PATH = "/content/drive/MyDrive/CropGuardAI/class_index.json"
SAVE_PATH = "/content/drive/MyDrive/CropGuardAI/cropguard_best.keras"

best_model = tf.keras.models.load_model(SAVE_PATH)
best_model.export(EXPORT_DIR)

idx = {str(v): k for k, v in train_gen.class_indices.items()}
with open(INDEX_PATH, "w") as f:
    json.dump(idx, f, indent=2)

print("Model exported to Google Drive!")
print("Download these two from Drive:")
print("  - CropGuardAI/plant_disease_model/  (folder)")
print("  - CropGuardAI/class_index.json")

Saved artifact at '/content/drive/MyDrive/CropGuardAI/plant_disease_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  137292749605968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740047312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740049424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740048272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740050384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740049808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740050000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740050576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740050192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137292740049616: TensorSpec(shape=(), dtype=